<style>
/* FABRIC notebook  adaptive theme */
.fab-info    { background-color: #f0f7fb; border-left: 4px solid #1f6a8c; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-success { background-color: #e8f5e9; border-left: 4px solid #008e7a; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-warning { background-color: #fff8e1; border-left: 4px solid #ff8542; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-danger  { background-color: #fce4ec; border-left: 4px solid #b00020; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-footer  { background-color: #374955; color: white; padding: 15px 20px; margin: 20px 0; border-radius: 4px; text-align: center; }

@media (prefers-color-scheme: dark) {
  .fab-info    { background-color: #1a2a35; border-color: #5798bc; color: #d0e0eb; }
  .fab-success { background-color: #1a2e25; border-color: #00b89a; color: #c0e0d5; }
  .fab-warning { background-color: #2e2518; border-color: #ff9a5c; color: #e0d0b8; }
  .fab-danger  { background-color: #2e1a1e; border-color: #e53950; color: #e0c0c8; }
  .fab-footer  { background-color: #2a3a45; color: #b0c4d0; }
}
</style>

# Sharing Slices Between FABRIC Project Members

<picture>
  <source srcset="../../images/fabric_logo_light.png" media="(prefers-color-scheme: dark)">
  <img src="../../images/fabric_logo.png" width="300" style="margin-bottom:10px;"/>
</picture>

<div class="fab-info">

**Welcome!** This notebook demonstrates how to **share FABRIC slices** with collaborators in your project. You will learn how to grant SSH access to team members, list slices created by others, and manage access control. This is a reference notebook that links to executable sub-notebooks for each operation.

</div>

## Learning Objectives

<div class="fab-success">

After reading this notebook you will understand:

1. How FABRIC's slice sharing model works within a project
2. How to grant a collaborator SSH access to VMs in your slice
3. How to discover and access slices created by other project members
4. How SSH keys (`sliver_key_name`) are used for access control
5. How to revoke access when collaboration ends

</div>

## Prerequisites

<div class="fab-warning">

Before using this guide:

1. Complete the [Configure Environment](../../../configure_and_validate/configure_and_validate.ipynb) notebook
2. Be a member of a FABRIC project with at least one other collaborator
3. Have an active slice you want to share (or know of a collaborator's slice to access)

**Also see:** The [Share Slices](../share_slices/share_slices.ipynb) notebook covers the same topic with links to sub-notebooks for SSH key management and project slice listing.

</div>

## Background: Slice Sharing on FABRIC

FABRIC organizes users into **projects**. Each slice belongs to a project, and by default only the slice creator can access its VMs. To collaborate, you need to explicitly grant SSH access to other project members.

### The Sharing Model

```
+-----------------------------------+
|           FABRIC Project          |
|                                   |
|  User A (Slice Owner)             |
|  - Creates Slice "experiment-1"   |
|  - Adds User B's SSH key          |
|                                   |
|  User B (Collaborator)            |
|  - Lists project slices           |
|  - Retrieves "experiment-1"       |
|  - SSHs into VMs                  |
+-----------------------------------+
```

### Key Concepts

| Concept | Description |
|---------|-------------|
| **Project** | A group of users who can share resources. All slices belong to a project. |
| **sliver_key_name** | The name of a user's SSH public key registered with FABRIC. Used to identify whose key to add. |
| **Slice visibility** | All project members can *see* project slices, but only users with SSH keys on VMs can *access* them. |
| **Access granularity** | SSH keys are added per-node, so you can grant access to specific VMs within a slice. |

### Two Sides of Sharing

**As a Slice Owner:**
1. Create your slice as usual
2. Get your collaborator's `sliver_key_name`
3. Add their SSH key to the relevant nodes
4. Remove the key when collaboration ends

**As a Collaborator:**
1. Register your SSH key with FABRIC (at the [portal](https://portal.fabric-testbed.net/))
2. Share your `sliver_key_name` with the slice owner
3. List project slices to find the shared slice
4. Retrieve the slice and SSH into the authorized VMs

## What We're Building

In this notebook we will create a single compute node with default resources.

<img src="./figs/slice_topology.png" width="40%">


---

## Granting Access to a Collaborator

To allow a team member to SSH into your slice's VMs, you add their SSH public key to each node they need access to.

### Using sliver_key_name (Recommended)

The most reliable method is to use the collaborator's **sliver_key_name**, which is the name of their SSH key registered with FABRIC:

```python
# Get the slice and node
slice = fablib.get_slice(name="my_experiment")
node = slice.get_node(name="Node1")

# Add collaborator's key by their sliver_key_name
node.add_public_key(sliver_key_name="collaborator_key_name")
```

### Using a Raw Public Key String

Alternatively, you can add a key directly:

```python
node.add_public_key(public_key="ssh-rsa AAAA... collaborator@example.com")
```

### Revoking Access

When collaboration ends, remove the key:

```python
node.remove_public_key(sliver_key_name="collaborator_key_name")
```

<div class="fab-danger">

**Important:** Only add SSH keys for users you trust. Anyone with SSH access to your VMs can execute arbitrary commands and access all data on those machines.

</div>

For detailed, executable instructions, see the [SSH Keys notebook](../share_slices/ssh_keys/ssh_keys.ipynb).

## Accessing a Collaborator's Slice

If a collaborator has added your SSH key to their slice, you can find and access it:

### List All Project Slices

```python
# List all slices in your project (including those created by others)
fablib.list_slices()
```

### Retrieve a Specific Slice

```python
# Get a slice by name (even if someone else created it)
slice = fablib.get_slice(name="collaborator_experiment")

# List nodes and their details
slice.list_nodes()

# SSH into a node
node = slice.get_node(name="Node1")
stdout, stderr = node.execute("hostname")
```

For detailed, executable instructions, see the [List Project Slices notebook](../share_slices/list_project_slices/list_project_slices.ipynb).

---

## Troubleshooting

| Problem | Possible Cause | Solution |
|---------|---------------|----------|
| Cannot see collaborator's slice | Not in the same project | Verify project membership at the [FABRIC portal](https://portal.fabric-testbed.net/) |
| SSH access denied to shared slice | Your SSH key not added to VMs | Ask the slice owner to add your `sliver_key_name` |
| `sliver_key_name` not found | Key not registered with FABRIC | Register your SSH key at the FABRIC portal |
| Cannot add collaborator's key | Incorrect key name or format | Verify the exact `sliver_key_name` with the collaborator |
| Access persists after key removal | SSH key cached in `authorized_keys` | Reboot the VM or manually edit `~/.ssh/authorized_keys` |
| `add_public_key()` fails | Slice not active or node unreachable | Verify the slice is in `StableOK` state |

## FABlib API Reference

| Method | Description | Documentation |
|--------|-------------|---------------|
| `fablib.list_slices()` | List all slices in your project | [list_slices](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.list_slices) |
| `fablib.get_slice(name)` | Retrieve a slice by name | [get_slice](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.get_slice) |
| `slice.get_nodes()` | Get all nodes in a slice | [get_nodes](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.get_nodes) |
| `slice.list_nodes()` | Display all nodes in a slice | [list_nodes](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.list_nodes) |
| `node.add_public_key(...)` | Add an SSH key to a node | [add_public_key](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.add_public_key) |
| `node.remove_public_key(...)` | Remove an SSH key from a node | [remove_public_key](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.remove_public_key) |
| `node.execute(command)` | Execute a command on a node | [execute](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.execute) |

## What's Next?

| Topic | Notebook | What You'll Learn |
|-------|----------|-------------------|
| **Share Slices (detailed)** | [share_slices](../share_slices/share_slices.ipynb) | Complete guide with sub-notebooks for SSH keys and listing |
| **SSH Keys Management** | [ssh_keys](../share_slices/ssh_keys/ssh_keys.ipynb) | Add and remove collaborator SSH keys |
| **List Project Slices** | [list_project_slices](../share_slices/list_project_slices/list_project_slices.ipynb) | Find and access shared slices |
| **Modify Slices** | [modify-add-node-network](../modify_slice/modify-add-node-network.ipynb) | Add and remove nodes and networks |
| **Hello FABRIC** | [hello_fabric](../hello_fabric/hello_fabric.ipynb) | Create your first slice |